# Tahap 1 - Membangun Case Base (CBR KDRT)
Notebook ini hanya untuk Tahap 1 sesuai instruksi tugas.

## 1. Install Library

In [9]:
import warnings
warnings.filterwarnings('ignore')

In [10]:
!pip -q install pdfplumber tqdm

In [11]:
import pdfplumber
print(f"pdfplumber version: {pdfplumber.__version__}")

pdfplumber version: 0.11.10


## 2. Mount Google Drive

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Konfigurasi Folder

In [19]:
import os
PDF_FOLDER='/content/drive/My Drive/TugasPK/Test/raw_pdf'

# Memastikan folder 'data/raw' dan 'logs' dibuat di dalam PDF_FOLDER
os.makedirs(os.path.join(PDF_FOLDER, 'data/raw'), exist_ok=True)
os.makedirs(os.path.join(PDF_FOLDER, 'logs'), exist_ok=True)

## 4. Ekstraksi PDF

In [4]:
import pdfplumber

def extract_pdf_text(pdf_path):
    text=''
    total_pages=0
    extracted_pages=0
    with pdfplumber.open(pdf_path) as pdf:
        total_pages=len(pdf.pages)
        for page in pdf.pages:
            t=page.extract_text()
            if t:
                extracted_pages+=1
                text+=t+'\n'
    return text,total_pages,extracted_pages

## 5. Cleaning

In [7]:
import re

def clean_text(text):
    text=re.sub(r'putusan\.mahkamahagung\.go\.id',' ',text,flags=re.I)
    text=re.sub(r'Mahkamah Agung Republik Indonesia',' ',text,flags=re.I)
    text=re.sub(r'Direktori Putusan Mahkamah Agung Republik Indonesia',' ',text,flags=re.I)
    text=re.sub(r'Hal\.\s*\d+.*',' ',text)
    text=text.lower()
    text=re.sub(r'[^a-z0-9\s]',' ',text)
    text=re.sub(r'\s+',' ',text)
    return text.strip()

## 6. PDF -> TXT

In [25]:
from tqdm import tqdm
import os
log_lines=[]
pdf_files=sorted([f for f in os.listdir(PDF_FOLDER) if f.lower().endswith('.pdf')])
for idx,pdf_file in enumerate(tqdm(pdf_files),start=1):
    text,total,ext=extract_pdf_text(os.path.join(PDF_FOLDER,pdf_file))
    cleaned=clean_text(text)
    # Menggunakan path lengkap ke Google Drive
    output_txt_path = os.path.join(PDF_FOLDER, 'data/raw', f'case_{idx:03d}.txt')
    with open(output_txt_path,'w',encoding='utf-8') as f:
        f.write(cleaned)
    completeness=(ext/total)*100 if total else 0
    status='SUCCESS' if completeness>=80 else 'WARNING'
    log_lines.append(f'{pdf_file}|{completeness:.2f}%|{status}')

100%|██████████| 30/30 [00:53<00:00,  1.78s/it]


## 7. Validasi

In [26]:
for x in log_lines:
    print(x)

Case_001.pdf|100.00%|SUCCESS
Case_002.pdf|100.00%|SUCCESS
Case_003.pdf|100.00%|SUCCESS
Case_004.pdf|100.00%|SUCCESS
Case_005.pdf|100.00%|SUCCESS
Case_006.pdf|100.00%|SUCCESS
Case_007.pdf|100.00%|SUCCESS
Case_008.pdf|100.00%|SUCCESS
Case_009.pdf|100.00%|SUCCESS
Case_010.pdf|100.00%|SUCCESS
Case_011.pdf|100.00%|SUCCESS
Case_012.pdf|100.00%|SUCCESS
Case_013.pdf|100.00%|SUCCESS
Case_014.pdf|100.00%|SUCCESS
Case_015.pdf|100.00%|SUCCESS
Case_016.pdf|100.00%|SUCCESS
Case_017.pdf|100.00%|SUCCESS
Case_018.pdf|100.00%|SUCCESS
Case_019.pdf|100.00%|SUCCESS
Case_020.pdf|100.00%|SUCCESS
Case_021.pdf|100.00%|SUCCESS
Case_022.pdf|100.00%|SUCCESS
Case_023.pdf|100.00%|SUCCESS
Case_024.pdf|100.00%|SUCCESS
Case_025.pdf|100.00%|SUCCESS
Case_026.pdf|100.00%|SUCCESS
Case_027.pdf|100.00%|SUCCESS
Case_028.pdf|100.00%|SUCCESS
Case_029.pdf|100.00%|SUCCESS
Case_030.pdf|100.00%|SUCCESS


## 8. Simpan cleaning.log

In [28]:
import os
# Menggunakan path lengkap ke Google Drive untuk menyimpan log
log_file_path = os.path.join(PDF_FOLDER, 'logs', 'cleaning.log')
with open(log_file_path,'w',encoding='utf-8') as f:
    f.write('\n'.join(log_lines))
print('selesai')

selesai
